In [8]:

import pandas as pd
import numpy as np
import time
import os
import glob
import gc
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

import kagglehub

# -------------------------------
# Download dataset
# -------------------------------
path = kagglehub.dataset_download(
    "bwandowando/spotify-songs-with-attributes-and-lyrics"
)

print("Path to dataset files:", path)

# -------------------------------
# Cari file CSV
# -------------------------------
csv_files = glob.glob(
    os.path.join(path, "**", "*.csv"),
    recursive=True
)

if not csv_files:
    raise FileNotFoundError(
        "File CSV tidak ditemukan pada folder dataset."
    )

print("\nFile CSV yang ditemukan:")
for i, file in enumerate(csv_files):
    size_mb = os.path.getsize(file) / (1024 ** 2)
    print(f"{i}: {os.path.basename(file)} ({size_mb:.2f} MB)")

# Pilih CSV pertama
csv_path = csv_files[0]

# -------------------------------
# Fitur audio yang digunakan
# -------------------------------
features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

# -------------------------------
# Periksa kolom tanpa membaca seluruh data
# -------------------------------
columns = pd.read_csv(csv_path, nrows=0).columns.tolist()

missing = [col for col in features if col not in columns]

if missing:
    raise ValueError(
        f"Kolom fitur tidak ditemukan: {missing}\n"
        f"Kolom tersedia: {columns}"
    )

# -------------------------------
# Load dataset dengan memori lebih hemat
# -------------------------------
MAX_ROWS = 50000  # Ubah menjadi None jika ingin membaca semua data

read_options = {
    "usecols": features,
    "dtype": {col: np.float32 for col in features},
    "engine": "c"
}

if MAX_ROWS is not None:
    read_options["nrows"] = MAX_ROWS

df = pd.read_csv(csv_path, **read_options)

# Pastikan semua fitur berupa numerik
df[features] = df[features].apply(
    pd.to_numeric, errors="coerce"
)

df = df.dropna(subset=features).reset_index(drop=True)

print("\nUkuran dataset setelah dibersihkan:", df.shape)
print("\nFitur yang digunakan:", features)
print(df.head())

# -------------------------------
# Siapkan matriks fitur
# -------------------------------
X = df[features].to_numpy(dtype=np.float32)

del df
gc.collect()

# -------------------------------
# Standardisasi fitur
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

n_samples, n_features = X_scaled.shape
k = 10

if n_samples < k:
    raise ValueError(
        f"Data hanya memiliki {n_samples} baris, "
        f"sedangkan k={k}."
    )

print("\nJumlah data:", n_samples)
print("Jumlah fitur:", n_features)
print("Ukuran matriks:", X_scaled.shape)

# -------------------------------
# Exact Nearest Neighbor
# -------------------------------
start = time.time()

nn = NearestNeighbors(
    n_neighbors=k,
    algorithm='brute',
    metric='euclidean',
    n_jobs=1
)

nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)

time_exact = time.time() - start

print(f"\nExact NN selesai: {time_exact:.3f} detik")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()

index_annoy = AnnoyIndex(n_features, 'euclidean')

for i, vector in enumerate(X_scaled):
    index_annoy.add_item(i, vector.tolist())

index_annoy.build(10)

idx_annoy = np.empty((n_samples, k), dtype=np.int32)

for i, vector in enumerate(X_scaled):
    idx_annoy[i] = index_annoy.get_nns_by_vector(
        vector.tolist(), k
    )

time_annoy = time.time() - start

print(f"Annoy selesai: {time_annoy:.3f} detik")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()

p_hnsw = hnswlib.Index(
    space='l2',
    dim=n_features
)

p_hnsw.init_index(
    max_elements=n_samples,
    ef_construction=200,
    M=16
)

p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)

idx_hnsw, dist_hnsw = p_hnsw.knn_query(
    X_scaled, k=k
)

time_hnsw = time.time() - start

print(f"HNSW selesai: {time_hnsw:.3f} detik")


# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()

quantizer = faiss.IndexFlatL2(n_features)

# Sesuaikan jumlah cluster dengan ukuran dataset
nlist = min(100, max(1, n_samples // 10))

# Gunakan argumen positional, bukan keyword metric
index_faiss = faiss.IndexIVFFlat(
    quantizer,
    n_features,
    nlist,
    faiss.METRIC_L2
)

index_faiss.train(X_scaled)
index_faiss.add(X_scaled)

index_faiss.nprobe = min(10, nlist)

dist_faiss, idx_faiss = index_faiss.search(
    X_scaled, k
)

time_faiss = time.time() - start

print(f"FAISS IVF selesai: {time_faiss:.3f} detik")


# -------------------------------
# Perbandingan waktu eksekusi
# -------------------------------
print("\n===== PERBANDINGAN WAKTU =====")
print(f"Exact NN : {time_exact:.3f} detik")
print(f"Annoy    : {time_annoy:.3f} detik")
print(f"HNSW     : {time_hnsw:.3f} detik")
print(f"FAISS IVF: {time_faiss:.3f} detik")

# -------------------------------
# Top-5 neighbors dari data pertama
# -------------------------------
print("\n===== TOP-5 NEIGHBORS DATA PERTAMA =====")
print("Exact NN :", idx_exact[0][:5])
print("Annoy    :", idx_annoy[0][:5])
print("HNSW     :", idx_hnsw[0][:5])
print("FAISS    :", idx_faiss[0][:5])


Path to dataset files: C:\Users\qulbiazzumi\.cache\kagglehub\datasets\bwandowando\spotify-songs-with-attributes-and-lyrics\versions\20

File CSV yang ditemukan:
0: songs_with_attributes_and_lyrics.csv (1473.56 MB)
1: songs_with_lyrics_and_timestamps.csv (2308.38 MB)

Ukuran dataset setelah dibersihkan: (50000, 9)

Fitur yang digunakan: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
   danceability  energy   loudness  speechiness  acousticness  \
0         0.415  0.6050 -11.157000       0.0575       0.00116   
1         0.788  0.6480  -9.135000       0.3150       0.90000   
2         0.000  0.0354 -20.150999       0.0000       0.90800   
3         0.657  0.8820  -6.340000       0.0385       0.00740   
4         0.659  0.8930  -8.531000       0.0411       0.09220   

   instrumentalness  liveness  valence       tempo  
0          0.838000    0.4710    0.193  100.058998  
1          0.000000    0.1760    0.287   79